## Chapter 15 - Operating on Data in Pandas

In [15]:
import pandas as pd
import numpy as np
rng = np.random.default_rng(42)

### Ufuncs: Index Preservation

In [16]:
ser = pd.Series(rng.integers(0, 10, 4))
df = pd.DataFrame(rng.integers(0, 10, (3, 4)), columns=['A', 'B', 'C', 'D'])

print(ser)
print(df)

0    0
1    7
2    6
3    4
dtype: int64
   A  B  C  D
0  4  8  0  6
1  2  0  5  9
2  7  7  7  7


- If we apply a NumPy `ufunc` on either of these objects, the result will be another Pandas object with the indices preserved:

In [17]:
print(np.exp(ser))
print(df * np.pi / 4)

0       1.000000
1    1096.633158
2     403.428793
3      54.598150
dtype: float64
          A         B         C         D
0  3.141593  6.283185  0.000000  4.712389
1  1.570796  0.000000  3.926991  7.068583
2  5.497787  5.497787  5.497787  5.497787


### Ufuncs: Index Alignment

- For binary operations on two Series or DataFrame objects, Pandas will align indices in the process of performing the operation. This is very convenient when working with incomplete data, as we’ll see in some of the examples that follow.

#### Index Alignment in Series

In [18]:
area = pd.Series({'Alaska': 1723337, 'Texas': 695662,  'California': 423967}, name='area')
population = pd.Series({'California': 39538223, 'Texas': 29145505,  'Florida': 21538187}, name='population')

print(population / area)

Alaska              NaN
California    93.257784
Florida             NaN
Texas         41.896072
dtype: float64


- If using `NaN` values is not the desired behavior, the fill value can be modified using appropriate object methods in place of the operators. For example, calling `A.add(B)` is equivalent to calling `A + B`, but allows optional explicit specification of the fill value for any elements in `A` or `B` that might be missing:

In [19]:
A = pd.Series([2, 4, 6], index=[0, 1, 2])
B = pd.Series([1, 3, 5], index=[1, 2, 3])

print(A + B)
print(A.add(B, fill_value=0))

0    NaN
1    5.0
2    9.0
3    NaN
dtype: float64
0    2.0
1    5.0
2    9.0
3    5.0
dtype: float64


#### Index Alignment in DataFrames

- A similar type of alignment takes place for both columns and indices when performing operations on `DataFrame` objects:

In [20]:
A = pd.DataFrame(rng.integers(0, 20, (2, 2)),  columns=['a', 'b'])
B = pd.DataFrame(rng.integers(0, 10, (3, 3)),  columns=['b', 'a', 'c'])

print(A)
print(B)

print(A + B)

    a  b
0  10  2
1  16  9
   b  a  c
0  5  3  1
1  9  7  6
2  4  8  5
      a     b   c
0  13.0   7.0 NaN
1  23.0  18.0 NaN
2   NaN   NaN NaN


### `Ufuncs`: Operations Between `DataFrames` and `Series`

- When performing operations between a `DataFrame` and a `Series`, the index and column alignment is similarly maintained, and the result is similar to operations between a two-dimensional and one-dimensional NumPy array. Consider one common operation, where we find the difference of a two-dimensional array and one of its rows:

In [21]:
A = rng.integers(10, size=(3, 4))
print(A)

print(A - A[0])

[[4 4 2 0]
 [5 8 0 8]
 [8 2 6 1]]
[[ 0  0  0  0]
 [ 1  4 -2  8]
 [ 4 -2  4  1]]


- In Pandas, the convention similarly operates row-wise by default:

In [22]:
df = pd.DataFrame(A, columns=['Q', 'R', 'S', 'T'])
df - df.iloc[0]

,Q,R,S,T
0,0,0,0,0
1,1,4,-2,8
2,4,-2,4,1


- If you would instead like to operate column-wise, you can use the object methods mentioned earlier, while specifying the axis keyword:

In [23]:
df.subtract(df['R'], axis=0)

,Q,R,S,T
0,0,0,-2,-4
1,-3,0,-8,0
2,6,0,4,-1
